In [1]:
import torch
from itertools import product

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import sys
root_dir = "/content/drive/MyDrive/Architectural-Biases-in-Time-Series-Anomaly-Detection"
sys.path.append(root_dir)
from app.utils.config import Config
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = Config(
    root_dir,
    "app",
    "data",
    "training_artifacts",
    "saved_model_weights",
    "training_results"
)
from app.training.train import fit
from app.models.transformer_encoder_forecaster import patch_transformer
from app.data.dataset import forecasting_Dataset
from app.scoring.knn_scorer import KNNResidualScorer
from app.evaluation.evaluate import evaluate_model

In [4]:
scorers = [KNNResidualScorer(device=device, n_components=n, k=k) for n, k in product([0.75, 0.85, 0.95], [3, 5, 7])]
def tune_over_grid(hyperparam_grid, exp_name, stop_at_epoch = 4, scorers = scorers):
    counter = 0
    for lr, lookback_window, d_model, num_blocks, horizon, num_heads, dropout in product(
        hyperparam_grid["lr"],
        hyperparam_grid["lookback_window"],
        hyperparam_grid["d_model"],
        hyperparam_grid["num_blocks"],
        hyperparam_grid["horizon"],
        hyperparam_grid["num_heads"],
        hyperparam_grid["dropout"]
    ):

        model = patch_transformer(
                    lookback_window, horizon, d_model,
                    num_heads, dropout, num_blocks
                ).to(device)
        model = torch.compile(model)

        name = (
            f"lr_{lr:.0e}_lw_{lookback_window}_dm_{d_model}_"
            f"nb_{num_blocks}_h_{horizon}_nh_{num_heads}_do_{dropout:.2f}"
        )

        forecast_train_dataset = forecasting_Dataset(
            device, cfg, lookback_window, horizon, start = 0, end = 800000
        )

        forecast_val_dataset = forecasting_Dataset(
            device, cfg, lookback_window, horizon, scaler = forecast_train_dataset.scaler,
            start = 800000, end = 1000000, train = False
        )

        fit(
            device=device,
            model=model,
            name=name,
            train_dataset=forecast_train_dataset,
            test_dataset=forecast_val_dataset,
            lr=lr,
            batch_size=512,
            num_epochs=10,
            stop_epoch_ratio=stop_at_epoch / 10.0,
            shuffle = True,
            patience = 5,
            min_delta = 0.0001
        )

        scoring_val_dataset = forecasting_Dataset(
            device, cfg, lookback_window, horizon, scaler = forecast_train_dataset.scaler,
            start = 1000000, end = 3500000, train = False
        )

        evaluate_model(device, model, forecast_val_dataset, scoring_val_dataset, exp_name + str(counter), cfg, scorers)
        counter+=1

### horizons

In [5]:
hyperparam_grid = {
    "lr": [0.001],
    "lookback_window": [256],
    "d_model": [256],
    "num_blocks": [1],
    "horizon": [4, 8, 16],
    "num_heads": [8],
    "dropout": [0.0]
}
exp_name = "exp_1_"
tune_over_grid(hyperparam_grid, exp_name)

32
LR Scheduler: 1093 warmup steps, 15620 total steps
|lr_1e-03_lw_256_dm_256_nb_1_h_4_nh_8_do_0.00| train = 0.1936 | test= 0.0213 | LR: 9.97e-04
|lr_1e-03_lw_256_dm_256_nb_1_h_4_nh_8_do_0.00| train = 0.0216 | test= 0.0206 | LR: 9.53e-04
|lr_1e-03_lw_256_dm_256_nb_1_h_4_nh_8_do_0.00| train = 0.0212 | test= 0.0205 | LR: 8.58e-04
|lr_1e-03_lw_256_dm_256_nb_1_h_4_nh_8_do_0.00| train = 0.0210 | test= 0.0206 | LR: 7.23e-04
--- evaluating model exp_1_0 --- 


OutOfMemoryError: CUDA out of memory. Tried to allocate 1860.03 GiB. GPU 0 has a total capacity of 79.25 GiB of which 77.30 GiB is free. Including non-PyTorch memory, this process has 1.94 GiB memory in use. Of the allocated memory 869.90 MiB is allocated by PyTorch, and 596.10 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### Lookbacks

In [ ]:
hyperparam_grid = {
    "lr": [0.001],
    "lookback_window": [128, 256, 384, 512],
    "d_model": [256],
    "num_blocks": [1],
    "horizon": [best_horizon],
    "num_heads": [8],
    "dropout": [0.0]
}
exp_name = "exp_2_"
tune_over_grid(hyperparam_grid, exp_name)

### d_model

In [ ]:
hyperparam_grid = {
    "lr": [0.001],
    "lookback_window": [best_lookback],
    "d_model": [128, 256, 384],
    "num_blocks": [1],
    "horizon": [best_horizon],
    "num_heads": [8],
    "dropout": [0.0]
}
exp_name = "exp_3_"
tune_over_grid(hyperparam_grid, exp_name)

### blocks

In [ ]:
hyperparam_grid = {
    "lr": [0.001],
    "lookback_window": [best_lookback],
    "d_model": [best_d_model],
    "num_blocks": [1, 2, 3],
    "horizon": [best_horizon],
    "num_heads": [8],
    "dropout": [0.0]
}
exp_name = "exp_4_"
tune_over_grid(hyperparam_grid, exp_name)

### heads

In [ ]:
hyperparam_grid = {
    "lr": [0.001],
    "lookback_window": [best_lookback],
    "d_model": [best_d_model],
    "num_blocks": [best_num_blocks],
    "horizon": [best_horizon],
    "num_heads": [4, 8, 16],
    "dropout": [0.0]
}
exp_name = "exp_5_"
tune_over_grid(hyperparam_grid, exp_name)

### lr and dropout

In [ ]:
hyperparam_grid = {
    "lr": [3e-4, 1e-3],
    "lookback_window": [best_lookback],
    "d_model": [best_d_model],
    "num_blocks": [best_num_blocks],
    "horizon": [best_horizon],
    "num_heads": [best_num_heads],
    "dropout": [0.0, 0.05, 0.1]
}
exp_name = "exp_6_"
tune_over_grid(hyperparam_grid, exp_name)